# Data ingestion learning

In [1]:
from pathlib import Path

from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials

SCOPES = ["https://www.googleapis.com/auth/gmail.readonly"]

project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

credentials_path = project_root / "credentials.json"
token_path = project_root / "token.json"

creds = None

if token_path.exists():
    creds = Credentials.from_authorized_user_file(token_path, SCOPES)

if not creds or not creds.valid:
    if creds and creds.expired and creds.refresh_token:
        creds.refresh(Request())
    else:
        flow = InstalledAppFlow.from_client_secrets_file(credentials_path, SCOPES)
        creds = flow.run_local_server(port=0)

    token_path.write_text(creds.to_json())

service = build("gmail", "v1", credentials=creds)


/Users/tommyzhao/Desktop/Projects/GmailRAG/.venv/lib/python3.9/site-packages/google/auth/__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
/Users/tommyzhao/Desktop/Projects/GmailRAG/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/tommyzhao/Desktop/Projects/GmailRAG/.venv/lib/python3.9/site-packages/google/oauth2/__init__.py:40: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes 

In [10]:
results = service.users().messages().list(
    userId="me",
    labelIds=["INBOX"],
    maxResults=10
).execute()

messages = results.get("messages", [])

print("Found messages:", len(messages))

for message in messages:
    print(message["id"])

Found messages: 10
19f2b7daee1127db
19f2b1e3c74d1fa4
19f2aa20523cace9
19f2a12072da0c85
19f29e5d0c659d68
19f29afc0ee51f8f
19f2977baa16ac3f
19f2958dd6919246
19f292b2bd5fa1cb
19f2926de0d43a28


In [11]:
import base64
from bs4 import BeautifulSoup

def decode_base64url(data):
    decoded_bytes = base64.urlsafe_b64decode(data + "===")
    return decoded_bytes.decode("utf-8", errors="replace")

def html_to_text(html):
    soup = BeautifulSoup(html, "html.parser")
    return soup.get_text(separator="\n", strip=True)

def collect_email_text_parts(payload, plain_text_parts, html_parts):
    mime_type = payload.get("mimeType")
    body_data = payload.get("body", {}).get("data")

    if body_data:
        decoded_text = decode_base64url(body_data)

        if mime_type == "text/plain":
            plain_text_parts.append(decoded_text)
        elif mime_type == "text/html":
            html_parts.append(html_to_text(decoded_text))

    for part in payload.get("parts", []):
        collect_email_text_parts(part, plain_text_parts, html_parts)

def extract_email_text(message):
    plain_text_parts = []
    html_parts = []

    collect_email_text_parts(
        message["payload"],
        plain_text_parts,
        html_parts,
    )

    if plain_text_parts:
        return "\n\n".join(plain_text_parts)

    if html_parts:
        return "\n\n".join(html_parts)

    return ""

In [12]:
def headers_to_dict(headers):
    return {header["name"].lower(): header["value"] for header in headers}

email_records = []

for message_summary in messages:
    full_message = service.users().messages().get(
        userId="me",
        id=message_summary["id"],
        format="full"
    ).execute()

    headers = headers_to_dict(full_message.get("payload", {}).get("headers", []))
    body = extract_email_text(full_message)

    email_record = {
        "id": full_message.get("id", ""),
        "thread_id": full_message.get("threadId", ""),
        "from": headers.get("from", ""),
        "to": headers.get("to", ""),
        "subject": headers.get("subject", ""),
        "date": headers.get("date", ""),
        "snippet": full_message.get("snippet", ""),
        "body": body,
    }

    email_records.append(email_record)

print("Extracted records:", len(email_records))
print("Records with readable body:", sum(1 for record in email_records if record["body"]))

Extracted records: 10
Records with readable body: 10


In [13]:
for index, record in enumerate(email_records, start=1):
    print("=" * 100)
    print(f"EMAIL {index}")
    print("Subject:", record["subject"])
    print("From:", record["from"])
    print("To:", record["to"])
    print("Date:", record["date"])
    print("Message ID:", record["id"])
    print()
    print(record["body"] if record["body"] else "[no readable body]")
    print()

EMAIL 1
Subject: 6 people noticed you
From: LinkedIn <messages-noreply@linkedin.com>
To: Tommy Zhao <tommyzhao898@gmail.com>
Date: Sat, 4 Jul 2026 04:58:04 +0000 (UTC)
Message ID: 19f2b7daee1127db

Your profile is looking great

Your work and accomplishments are being recognized

6 profile views
  
          https://www.linkedin.com/comm/me/profile-views?lipi=urn%3Ali%3Apage%3Aemail_b2_professional_identity_digest_02%3BuAAk4Pz9RJqJeeYyHVDVVg%3D%3D&midToken=AQHEwR7n3aC-ig&midSig=2BGLQjVSzGusk1&trk=eml-b2_professional_identity_digest_02-email~professional~identity~digest-0-profile~views&trkEmail=eml-b2_professional_identity_digest_02-email~professional~identity~digest-0-profile~views-null-ohrhx5~mr5w5tri~ee-null-null&eid=ohrhx5-mr5w5tri-ee&otpToken=MTMwMTE2ZTExMjJjY2RjMmI3MmQwZmViNDExNmU0YjU4YmNmZDA0OTlkYWU4NzZiN2JjZTA2Njg0YzUyNWVmNGY1ZDZkMWU5NjFlYWNlZjQ2MDlmZjA4MmVkZTNmZTNhZWJiODliYzQ1NDMzN2NhODViY2QyYTZlLDEsMQ%3D%3D

Unlock the full list with Premium
Premium subscribers get 11x more pr